# Decoding


In [ ]:
import math
import zlib
import json
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch
import torch.nn.functional as F

import import_ipynb
from model_architecture import ASRModel, CONFIGS
from tokenizer_training import MultitaskTokenizer, BPETokenizer, base_alphabet, merges
from audio_frontend import log_mel_spectrogram, pad_or_trim, N_SAMPLES, SAMPLE_RATE, HOP_LENGTH

In [ ]:
NO_SPEECH_THRESHOLD = 0.6
LOGPROB_THRESHOLD = -1.0
COMPRESSION_RATIO_THRESHOLD = 2.4
TEMPERATURES = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
MAX_DECODE_TOKENS = 224

@dataclass
class DecodeOptions:
    language: Optional[str] = None
    task: str = "transcribe"
    timestamps: bool = True
    beam_size: Optional[int] = None
    patience: float = 1.0

@dataclass
class Segment:
    start: float
    end: float
    text: str
    tokens: list
    avg_logprob: float
    no_speech_prob: float
    temperature: float
    language: str

In [ ]:
def compression_ratio(text):
    data = text.encode("utf-8")
    if not data:
        return 0.0
    return len(data) / len(zlib.compress(data))

def detect_language(model, tokenizer, mel):
    with torch.no_grad():
        audio = model.encoder(mel[None])
        sot = torch.tensor([[tokenizer.sot]], device=mel.device)
        logits = model.decoder(sot, audio)[0, 0]
    mask = torch.full_like(logits, float("-inf"))
    for lang, idx in tokenizer.language_ids.items():
        mask[idx] = 0.0
    probs = (logits + mask).softmax(dim=-1)
    best = int(probs.argmax())
    lang = next(l for l, i in tokenizer.language_ids.items() if i == best)
    return lang, float(probs[best])

In [ ]:
class SuppressBlank:
    def __init__(self, tokenizer, sample_begin):
        self.blank_ids = [tokenizer.bpe.vocab.get("\u0120", -1)]
        self.eot = tokenizer.eot
        self.sample_begin = sample_begin

    def __call__(self, logits, tokens):
        if tokens.shape[1] == self.sample_begin:
            for b in self.blank_ids:
                if b >= 0:
                    logits[:, b] = float("-inf")
            logits[:, self.eot] = float("-inf")
        return logits

class TimestampRules:
    def __init__(self, tokenizer, sample_begin, max_initial=1.0):
        self.tokenizer = tokenizer
        self.sample_begin = sample_begin
        self.max_initial_token = tokenizer.timestamp_token(max_initial)

    def __call__(self, logits, tokens):
        tk = self.tokenizer
        for b in range(tokens.shape[0]):
            seq = tokens[b, self.sample_begin:].tolist()
            last_was_timestamp = len(seq) >= 1 and seq[-1] >= tk.timestamp_begin
            penultimate_was = len(seq) >= 2 and seq[-2] >= tk.timestamp_begin
            if last_was_timestamp:
                if penultimate_was:
                    logits[b, tk.timestamp_begin:] = float("-inf")
                else:
                    logits[b, : tk.eot] = float("-inf")
            stamps = [t for t in seq if t >= tk.timestamp_begin]
            if stamps:
                logits[b, tk.timestamp_begin: stamps[-1]] = float("-inf")
            if len(seq) == 0:
                logits[b, tk.eot] = float("-inf")
                logits[b, self.max_initial_token + 1:] = float("-inf")
            probs = logits[b].float().log_softmax(dim=-1)
            timestamp_lp = probs[tk.timestamp_begin:].logsumexp(dim=-1)
            max_text_lp = probs[: tk.timestamp_begin].max()
            if timestamp_lp > max_text_lp:
                logits[b, : tk.timestamp_begin] = float("-inf")
        return logits

In [ ]:
def greedy_decode(model, tokenizer, audio_features, prefix, temperature, options, generator=None):
    device = audio_features.device
    tokens = torch.tensor([prefix], device=device)
    sample_begin = len(prefix)
    filters = [SuppressBlank(tokenizer, sample_begin)]
    if options.timestamps:
        filters.append(TimestampRules(tokenizer, sample_begin))
    sum_logprob = 0.0
    no_speech_prob = None
    kv_cache = {}
    hooks = []
    def make_hook(module):
        def hook(mod, inp, out):
            if mod in kv_cache and out.shape[1] == 1:
                kv_cache[mod] = torch.cat([kv_cache[mod], out], dim=1)
            else:
                kv_cache[mod] = out
            return kv_cache[mod]
        return hook
    for block in model.decoder.blocks:
        hooks.append(block.attn.key.register_forward_hook(make_hook(block.attn.key)))
        hooks.append(block.attn.value.register_forward_hook(make_hook(block.attn.value)))
    try:
        with torch.no_grad():
            next_input = tokens
            for step in range(MAX_DECODE_TOKENS):
                logits = model.decoder(next_input, audio_features, kv_cache=kv_cache)[:, -1]
                if step == 0:
                    probs = logits.float().softmax(dim=-1)
                    no_speech_prob = float(probs[0, tokenizer.no_speech])
                for f in filters:
                    logits = f(logits, tokens)
                if temperature == 0.0:
                    next_token = logits.argmax(dim=-1)
                else:
                    probs = (logits / temperature).float().softmax(dim=-1)
                    next_token = torch.multinomial(probs, 1, generator=generator).squeeze(-1)
                lp = logits.float().log_softmax(dim=-1)
                sum_logprob += float(lp[0, next_token[0]])
                tokens = torch.cat([tokens, next_token[:, None]], dim=1)
                next_input = next_token[:, None]
                if int(next_token[0]) == tokenizer.eot:
                    break
    finally:
        for h in hooks:
            h.remove()
    decoded = tokens[0, sample_begin:].tolist()
    if decoded and decoded[-1] == tokenizer.eot:
        decoded = decoded[:-1]
    avg_logprob = sum_logprob / max(len(decoded) + 1, 1)
    return decoded, avg_logprob, no_speech_prob

In [ ]:
def decode_with_fallback(model, tokenizer, audio_features, prefix, options, seed):
    best_tokens, best_lp, best_nsp = [], float("-inf"), 1.0
    last = None
    for attempt, temperature in enumerate(TEMPERATURES):
        generator = torch.Generator(device=audio_features.device)
        generator.manual_seed(seed ^ (attempt << 32))
        tokens, avg_lp, nsp = greedy_decode(
            model, tokenizer, audio_features, prefix, temperature, options, generator
        )
        text = tokenizer.bpe.decode([t for t in tokens if t < tokenizer.eot])
        repetitive = compression_ratio(text) > COMPRESSION_RATIO_THRESHOLD
        if not repetitive and (avg_lp > best_lp or not best_tokens):
            best_tokens, best_lp, best_nsp = tokens, avg_lp, nsp
        last = (tokens, avg_lp, nsp, temperature)
        needs_fallback = repetitive or avg_lp < LOGPROB_THRESHOLD
        is_silence = nsp > NO_SPEECH_THRESHOLD and avg_lp < LOGPROB_THRESHOLD
        if not needs_fallback or is_silence:
            return (best_tokens or tokens), (best_lp if best_tokens else avg_lp), nsp, temperature
    if not best_tokens and last is not None:
        return last
    return best_tokens, best_lp, best_nsp, TEMPERATURES[-1]

In [ ]:
def transcribe(model, tokenizer, wave, options=DecodeOptions()):
    device = next(model.parameters()).device
    segments = []
    offset = 0
    total = len(wave)
    language = options.language
    while offset < total:
        chunk = wave[offset: offset + N_SAMPLES]
        chunk_seconds = len(chunk) / SAMPLE_RATE
        mel = log_mel_spectrogram(pad_or_trim(torch.as_tensor(chunk, dtype=torch.float32))).to(device)
        with torch.no_grad():
            audio_features = model.encoder(mel[None])
        if language is None:
            language, _ = detect_language(model, tokenizer, mel)
        prefix = tokenizer.sot_sequence(language, options.task, timestamps=options.timestamps)
        tokens, avg_lp, nsp, temperature = decode_with_fallback(
            model, tokenizer, audio_features, prefix, options, seed=offset
        )
        if nsp > NO_SPEECH_THRESHOLD and avg_lp < LOGPROB_THRESHOLD:
            offset += N_SAMPLES
            continue
        base = offset / SAMPLE_RATE
        stamps = [i for i, t in enumerate(tokens) if t >= tokenizer.timestamp_begin]
        if options.timestamps and len(stamps) >= 2:
            for a, b in zip(stamps[::2], stamps[1::2]):
                start = min(tokenizer.timestamp_seconds(tokens[a]), chunk_seconds)
                end = min(tokenizer.timestamp_seconds(tokens[b]), chunk_seconds)
                text_tokens = [t for t in tokens[a + 1: b] if t < tokenizer.eot]
                text = tokenizer.bpe.decode(text_tokens).strip()
                if text:
                    segments.append(Segment(base + start, base + end, text, text_tokens,
                                            avg_lp, nsp, temperature, language))
            advance = tokenizer.timestamp_seconds(tokens[stamps[-1]])
            offset += max(int(advance * SAMPLE_RATE), HOP_LENGTH)
        else:
            text_tokens = [t for t in tokens if t < tokenizer.eot]
            text = tokenizer.bpe.decode(text_tokens).strip()
            if text:
                segments.append(Segment(base, base + chunk_seconds, text, text_tokens,
                                        avg_lp, nsp, temperature, language))
            offset += N_SAMPLES
    return segments

In [ ]:
bpe = BPETokenizer(base_alphabet, merges)
tokenizer = MultitaskTokenizer(bpe)
dims = CONFIGS["tiny"]
dims.n_vocab = tokenizer.n_vocab
model = ASRModel(dims)
state = torch.load("runs/tiny-multitask-v1/step-final.pt", map_location="cpu", weights_only=False)
model.load_state_dict(state["model"])
model.eval()

In [ ]:
import soundfile as sf

wave, sr = sf.read("data/samples/meeting-excerpt.wav", dtype="float32")
if wave.ndim > 1:
    wave = wave.mean(axis=1)
segments = transcribe(model, tokenizer, wave)
for s in segments:
    print(f"[{s.start:7.2f} {s.end:7.2f}] lp={s.avg_logprob:6.2f} nsp={s.no_speech_prob:.2f} t={s.temperature:.1f}  {s.text}")

In [ ]:
silence = np.zeros(SAMPLE_RATE * 10, dtype=np.float32)
print(len(transcribe(model, tokenizer, silence)))

noise = (np.random.default_rng(0).standard_normal(SAMPLE_RATE * 10) * 0.05).astype(np.float32)
print(len(transcribe(model, tokenizer, noise)))

In [ ]:
def evidence_report(segments):
    if not segments:
        return {"segments": 0}
    lps = [s.avg_logprob for s in segments]
    return {
        "segments": len(segments),
        "avg_logprob_mean": round(float(np.mean(lps)), 3),
        "avg_logprob_min": round(float(np.min(lps)), 3),
        "fallback_segments": sum(1 for s in segments if s.temperature > 0),
        "low_confidence_segments": sum(1 for s in segments if s.avg_logprob < LOGPROB_THRESHOLD),
        "languages": sorted({s.language for s in segments}),
    }

print(json.dumps(evidence_report(segments), indent=2))